# Датасеты и контроль утечек
Воспроизводимый обзор фактических файлов. Исходные данные и прежний ноутбук не изменяются.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'ecg_project').exists(): ROOT = ROOT.parent
os.chdir(ROOT)
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))


In [2]:
catalog = pd.read_csv('artifacts/catalog.csv').fillna('')
catalog.groupby('source').agg(records=('record_id','size'), readable=('readable','sum'), labeled=('has_labels','sum'), min_seconds=('duration_s','min'), max_seconds=('duration_s','max'))

,records,readable,labeled,min_seconds,max_seconds
source,,,,,
LUDB,200,200,0,10.0,10.0
Training_2,3453,3452,2433,8.0,98.0
Training_StPetersburg,74,74,74,1800.0,1800.0
Training_WFDB,6877,6877,6877,6.0,144.0
WFDB_GEORGIA,10344,10344,10344,5.0,10.0
WFDB_PTB-XL,21837,21837,21837,10.0,10.0


Отсутствие #Dx означает неизвестную метку. Типы сокращений MIT-BIH не являются диагнозами всей записи. LUDB и data/ludb_beats — один источник, не независимые выборки.

In [3]:
from ecg_project.data.catalog import TARGETS
catalog[catalog.has_labels].groupby('source')[list(TARGETS)].sum()

,AF,PVC,SVEB,VT,AVB1,AVB2,AVB3,AVB,IVCD
source,,,,,,,,,
Training_2,115,134,90,1,73,20,20,118,163
Training_StPetersburg,3,23,7,16,0,3,0,3,4
Training_WFDB,1221,0,616,0,722,0,0,722,2093
WFDB_GEORGIA,570,357,640,0,769,23,8,870,1398
WFDB_PTB-XL,1514,0,555,0,797,14,16,827,3045


In [4]:
from ecg_project.data.catalog import ludb_split, assert_disjoint
from ecg_project.training.beats import split_for
assert split_for('201') == split_for('202')
splits = ludb_split(range(1,201))
pd.Series(splits).value_counts()

train    140
test      30
valid     30
Name: count, dtype: int64

In [5]:
f = pd.read_csv('artifacts/record_features_qt/manifest.csv').fillna('')
assert_disjoint(f)
f.groupby(['source','split']).size().unstack(fill_value=0)

split,external,external_long,test,train,valid
source,,,,,
Training_2,0,0,0,625,0
Training_StPetersburg,0,74,0,0,0
Training_WFDB,0,0,0,497,0
WFDB_GEORGIA,499,0,0,0,0
WFDB_PTB-XL,0,0,500,524,499


Основной PTB-XL split использует официальные patient_id. Georgia целиком внешний источник; совпадения пациентов между источниками исключены по подтверждению владельца данных. Пилот ограничен 500 записями на source/split, с сохранением всех редких train-позитивов. Это не полное обучение на всём архиве.

In [6]:
from ecg_project.data.io import load_record, annotations
r=load_record('LUDB/1.hea')
print(r.fs, r.signal.shape, r.leads, r.unit)
pd.DataFrame(annotations('LUDB/1.hea','II')).head()

500.0 (5000, 12) ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'] mV


,wave,onset,peak,offset,lead,source
0,QRS,644,662,682,II,annotation
1,T,776,843,878,II,annotation
2,P,1250,1278,1302,II,annotation
3,QRS,1324,1342,1374,II,annotation
4,T,1458,1524,1572,II,annotation


Сведения о спорной калибровке PTB-XL/StPetersburg и изменённых заголовках Training_2: ../reports/research_notes.md. Амплитуды из спорной калибровки не используются как физические признаки.